# Products Bronze — Production-style ingestion

Supports `INCREMENTAL`, `BACKFILL`, and `FULL_LOAD` for JSON files named `products_YYYYMMDD.json`.

**Safety:** run this notebook with one writer for this dataset. The notebook only writes the Products manifest and Products Bronze paths configured below. It does not touch Orders or the isolated test manifest. Review configuration and run mode before execution.


In [0]:

# CELL 1 — CONFIGURATION AND WIDGETS
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType, DoubleType
)
from datetime import datetime, timezone, timedelta
import uuid

DATASET_NAME = "products"
SOURCE_SYSTEM = "product_api"
STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"
BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
LANDING_PATH = f"{BASE_PATH}landing/{DATASET_NAME}/"
BRONZE_PATH = f"{BASE_PATH}bronze/{DATASET_NAME}/"
MANIFEST_PATH = f"{BASE_PATH}metadata/file_manifest/"
LEASE_MINUTES = 60

PRODUCTS_SCHEMA = StructType([
    StructField("product_id", LongType(), nullable=False),
    StructField("product_name", StringType(), nullable=True),
    StructField("category", StringType(), nullable=True),
    StructField("brand", StringType(), nullable=True),
    StructField("price", DoubleType(), nullable=True),
])

for name in ["run_mode", "backfill_start_date", "backfill_end_date", "reprocess_successful", "dry_run"]:
    try:
        dbutils.widgets.remove(name)
    except Exception:
        pass

dbutils.widgets.dropdown("run_mode", "INCREMENTAL", ["INCREMENTAL", "BACKFILL", "FULL_LOAD"], "Run Mode")
dbutils.widgets.text("backfill_start_date", "", "Backfill Start Date (YYYY-MM-DD)")
dbutils.widgets.text("backfill_end_date", "", "Backfill End Date (YYYY-MM-DD)")
dbutils.widgets.dropdown("reprocess_successful", "false", ["false", "true"], "Reprocess Successful Files?")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "Dry Run? (true = no writes)")

RUN_MODE = dbutils.widgets.get("run_mode").strip().upper()
BACKFILL_START_DATE = dbutils.widgets.get("backfill_start_date").strip()
BACKFILL_END_DATE = dbutils.widgets.get("backfill_end_date").strip()
REPROCESS_SUCCESSFUL = dbutils.widgets.get("reprocess_successful").strip().lower() == "true"
DRY_RUN = dbutils.widgets.get("dry_run").strip().lower() == "true"

assert RUN_MODE in {"INCREMENTAL", "BACKFILL", "FULL_LOAD"}
if RUN_MODE == "BACKFILL":
    assert BACKFILL_START_DATE and BACKFILL_END_DATE, "BACKFILL requires both dates."
    start_date = datetime.strptime(BACKFILL_START_DATE, "%Y-%m-%d").date()
    end_date = datetime.strptime(BACKFILL_END_DATE, "%Y-%m-%d").date()
    assert start_date <= end_date, "Backfill start date must be <= end date."

print("Mode:", RUN_MODE, "| dry_run:", DRY_RUN, "| reprocess_successful:", REPROCESS_SUCCESSFUL)
print("Landing:", LANDING_PATH)
print("Bronze:", BRONZE_PATH)
print("Manifest:", MANIFEST_PATH)


Mode: FULL_LOAD | dry_run: True | reprocess_successful: False
Landing: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/
Bronze: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/
Manifest: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/file_manifest/


In [0]:

# CELL 2 — DISCOVER FILES AND CALCULATE SHA-256 (READ-ONLY)
def list_files_recursive(path):
    found = []
    for item in dbutils.fs.ls(path):
        if item.isDir():
            found.extend(list_files_recursive(item.path))
        else:
            found.append((item.name, item.path, int(item.size)))
    return found

discovered = list_files_recursive(LANDING_PATH)
landing_schema = StructType([
    StructField("file_name", StringType(), False),
    StructField("file_path", StringType(), False),
    StructField("file_size_bytes", LongType(), False),
])
landing_df = spark.createDataFrame(discovered, landing_schema) if discovered else spark.createDataFrame([], landing_schema)

dated_df = (
    landing_df
    .withColumn("date_string", F.regexp_extract("file_name", r"^products_(\d{8})\.json$", 1))
    .withColumn("data_date", F.to_date("date_string", "yyyyMMdd"))
    .withColumn("filename_status", F.when(F.col("date_string") == "", "INVALID_FILENAME")
                .when(F.col("data_date").isNull(), "INVALID_DATE").otherwise("VALID"))
)

invalid = dated_df.filter(F.col("filename_status") != "VALID").count()
assert invalid == 0, "Invalid file names/dates found. Fix them before continuing."

binary_df = (
    spark.read.format("binaryFile").load(LANDING_PATH + "**")
    .select(F.col("path").alias("file_path"), F.sha2("content", 256).alias("actual_content_hash"))
)
files_df = dated_df.filter(F.col("filename_status") == "VALID").join(binary_df, "file_path", "left")
assert files_df.filter(F.col("actual_content_hash").isNull()).count() == 0, "A file could not be hashed."
display(files_df.select("file_name", "data_date", "file_size_bytes", "actual_content_hash").orderBy("data_date"))
print("Discovered files:", files_df.count())


file_name,data_date,file_size_bytes,actual_content_hash
products_20261007.json,2026-10-07,3662500,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118
products_20261008.json,2026-10-08,2615396,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b


Discovered files: 2


In [0]:

# CELL 3 — RECONCILE AND SELECT FILES (READ-ONLY)
manifest_all_df = spark.read.format("delta").load(MANIFEST_PATH)
manifest_df = (
    manifest_all_df.filter(F.col("dataset") == DATASET_NAME)
    .select("dataset", "file_name", "status", "content_hash", "data_date",
            "retry_count", "attempt_count", "run_id")
)
duplicates = manifest_df.groupBy("dataset", "file_name").count().filter("count > 1").count()
assert duplicates == 0, "Duplicate Products manifest keys found."

reconciliation_df = (
    files_df.alias("l")
    .join(manifest_df.alias("m"), F.col("l.file_name") == F.col("m.file_name"), "left")
    .select(
        F.col("l.file_name"), F.col("l.file_path"), F.col("l.file_size_bytes"),
        F.col("l.data_date").alias("landing_data_date"), F.col("l.actual_content_hash"),
        F.col("m.status").alias("manifest_status"), F.col("m.content_hash").alias("manifest_hash"),
        F.col("m.data_date").alias("manifest_data_date")
    )
    .withColumn("effective_data_date", F.coalesce("manifest_data_date", "landing_data_date"))
    .withColumn(
        "selection_reason",
        F.when(F.col("manifest_status") == "PROCESSING", "SKIP_PROCESSING")
         .when(F.col("manifest_status").isNull(), "REGISTER_REQUIRED")
         .when(F.col("manifest_hash").isNull(), "REVIEW_REQUIRED")
         .when(F.col("manifest_hash") != F.col("actual_content_hash"), "CONTENT_CHANGED")
         .when(F.col("manifest_status").isin("NEW", "FAILED"), "ELIGIBLE")
         .when(F.col("manifest_status") == "SUCCESS", "SUCCESS_REPROCESS_CANDIDATE")
         .otherwise("REVIEW_REQUIRED")
    )
)

base_reasons = ["REGISTER_REQUIRED", "ELIGIBLE", "CONTENT_CHANGED"]
if RUN_MODE == "INCREMENTAL":
    selected_df = reconciliation_df.filter(F.col("selection_reason").isin(base_reasons))
elif RUN_MODE == "BACKFILL":
    reasons = base_reasons + (["SUCCESS_REPROCESS_CANDIDATE"] if REPROCESS_SUCCESSFUL else [])
    selected_df = reconciliation_df.filter(
        F.col("effective_data_date").between(
            F.to_date(F.lit(BACKFILL_START_DATE)), F.to_date(F.lit(BACKFILL_END_DATE))
        ) & F.col("selection_reason").isin(reasons)
    )
else:  # FULL_LOAD includes successful files regardless of reprocess_successful
    selected_df = reconciliation_df.filter(
        F.col("selection_reason").isin(base_reasons + ["SUCCESS_REPROCESS_CANDIDATE"])
    )

display(reconciliation_df.select("file_name", "effective_data_date", "manifest_status",
                                  "selection_reason").orderBy("effective_data_date"))
print("Selected candidates:", selected_df.count())


file_name,effective_data_date,manifest_status,selection_reason
products_20261007.json,2026-10-07,SUCCESS,SUCCESS_REPROCESS_CANDIDATE
products_20261008.json,2026-10-08,SUCCESS,SUCCESS_REPROCESS_CANDIDATE


Selected candidates: 2


In [0]:

# CELL 4 — SAFETY GATE AND DRY-RUN PLAN (READ-ONLY)
selected_count = selected_df.count()
duplicate_selected = selected_df.groupBy("file_name").count().filter("count > 1").count()
bad_hashes = selected_df.filter(F.col("actual_content_hash").isNull() | (F.length("actual_content_hash") != 64)).count()
bad_dates = selected_df.filter(F.col("effective_data_date").isNull()).count()
processing_selected = selected_df.filter(F.col("manifest_status") == "PROCESSING").count()
review_selected = selected_df.filter(F.col("selection_reason") == "REVIEW_REQUIRED").count()

assert duplicate_selected == 0
assert bad_hashes == 0
assert bad_dates == 0
assert processing_selected == 0
assert review_selected == 0

display(selected_df.select("file_name", "file_path", "effective_data_date",
                           "manifest_status", "selection_reason", "actual_content_hash"))
print("Safety gate PASS | selected:", selected_count, "| dry_run:", DRY_RUN)
if DRY_RUN:
    print("DRY RUN: no manifest or Bronze writes will occur. Set dry_run=false only after reviewing this plan.")


file_name,file_path,effective_data_date,manifest_status,selection_reason,actual_content_hash
products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,2026-10-07,SUCCESS,SUCCESS_REPROCESS_CANDIDATE,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118
products_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=08/products_20261008.json,2026-10-08,SUCCESS,SUCCESS_REPROCESS_CANDIDATE,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b


Safety gate PASS | selected: 2 | dry_run: True
DRY RUN: no manifest or Bronze writes will occur. Set dry_run=false only after reviewing this plan.


In [0]:

# CELL 5 — PROCESS SELECTED FILES
# This cell writes only when dry_run=false. Use one writer for this dataset.
if DRY_RUN:
    print("DRY RUN complete. No writes performed.")
else:
    if selected_df.filter(F.col("selection_reason") == "REGISTER_REQUIRED").limit(1).count():
        raise RuntimeError(
            "Selected file(s) are not registered in the manifest. Register new files using your approved manifest-registration workflow, "
            "then rerun discovery. This notebook intentionally refuses to invent manifest defaults."
        )

    run_id = str(uuid.uuid4())
    print("Processing run:", run_id)
    results = []

    for candidate in selected_df.orderBy("effective_data_date", "file_name").collect():
        file_name = candidate["file_name"]
        file_path = candidate["file_path"]
        actual_hash = candidate["actual_content_hash"]
        expected_status = candidate["manifest_status"]
        old_hash = candidate["manifest_hash"] if "manifest_hash" in candidate else None

        # Refresh manifest row and verify it is still eligible.
        current = (
            spark.read.format("delta").load(MANIFEST_PATH)
            .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
            .select("status", "content_hash", "attempt_count", "retry_count")
            .collect()
        )
        assert len(current) == 1, f"{file_name}: expected exactly one manifest row."
        current = current[0]
        assert current["status"] != "PROCESSING", f"{file_name}: another run is processing this file."
        assert current["status"] == expected_status, f"{file_name}: manifest status changed; rerun discovery."
        assert current["content_hash"] == (candidate["manifest_hash"]), f"{file_name}: manifest hash changed; rerun discovery."

        # Verify source has not changed since discovery.
        fresh_hash = (
            spark.read.format("binaryFile").load(file_path)
            .select(F.sha2("content", 256).alias("h")).first()["h"]
        )
        assert fresh_hash == actual_hash, f"{file_name}: source changed since discovery."

        batch_id = str(uuid.uuid4())
        now = datetime.now(timezone.utc)
        lease_expiry = now + timedelta(minutes=LEASE_MINUTES)
        manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)

        # Claim record conditionally. If content changed, update manifest hash to the newly discovered hash.
        claim_condition = (
            (F.col("t.dataset") == DATASET_NAME) &
            (F.col("t.file_name") == file_name) &
            (F.col("t.status") == expected_status) &
            (F.col("t.content_hash") == candidate["manifest_hash"])
        )
        manifest_delta.alias("t").update(
            condition=claim_condition,
            set={
                "status": F.lit("PROCESSING"),
                "run_id": F.lit(run_id),
                "claimed_at": F.lit(now).cast("timestamp"),
                "lease_expires_at": F.lit(lease_expiry).cast("timestamp"),
                "last_attempt_at": F.lit(now).cast("timestamp"),
                "attempt_count": F.coalesce(F.col("t.attempt_count"), F.lit(0)) + F.lit(1),
                "status_updated_at": F.lit(now).cast("timestamp"),
                "content_hash": F.lit(actual_hash),
                "error_message": F.lit(None).cast("string"),
                "error_type": F.lit(None).cast("string"),
            }
        )

        owner = (
            spark.read.format("delta").load(MANIFEST_PATH)
            .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
            .select("status", "run_id", "content_hash").first()
        )
        assert owner and owner["status"] == "PROCESSING" and owner["run_id"] == run_id, f"{file_name}: claim failed."
        assert owner["content_hash"] == actual_hash, f"{file_name}: claimed hash mismatch."

        try:
            source_df = (
                spark.read.schema(PRODUCTS_SCHEMA)
                .option("multiLine", "true")
                .option("mode", "FAILFAST")
                .json(file_path)
            )
            source_count = source_df.count()
            null_ids = source_df.filter(F.col("product_id").isNull()).count()
            distinct_ids = source_df.select("product_id").distinct().count()
            negative_prices = source_df.filter(F.col("price") < 0).count()
            assert source_count > 0, f"{file_name}: source is empty."
            assert null_ids == 0, f"{file_name}: null product IDs found."
            assert distinct_ids == source_count, f"{file_name}: duplicate product IDs found within file."
            assert negative_prices == 0, f"{file_name}: negative prices found."

            ingested_at = datetime.now(timezone.utc).replace(tzinfo=None)
            replacement_df = (
                source_df
                .withColumn("_source_file", F.lit(file_name))
                .withColumn("_source_path", F.lit(file_path))
                .withColumn("_dataset", F.lit(DATASET_NAME))
                .withColumn("_batch_id", F.lit(batch_id))
                .withColumn("_run_id", F.lit(run_id))
                .withColumn("_ingested_at", F.lit(ingested_at).cast("timestamp"))
                .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
            )

            # Existing Bronze table must exist; replace only this file's partition/rows.
            bronze_before = spark.read.format("delta").load(BRONZE_PATH)
            target_before = bronze_before.filter(F.col("_source_file") == file_name).count()
            (replacement_df.write.format("delta").mode("overwrite")
             .option("replaceWhere", f"_source_file = '{file_name}'")
             .save(BRONZE_PATH))

            post_target = spark.read.format("delta").load(BRONZE_PATH).filter(F.col("_source_file") == file_name)
            written_count = post_target.count()
            post_nulls = post_target.filter(F.col("product_id").isNull()).count()
            wrong_run = post_target.filter(F.col("_run_id") != run_id).count()
            assert written_count == source_count, f"{file_name}: written count mismatch."
            assert post_nulls == 0 and wrong_run == 0, f"{file_name}: post-write validation failed."

            version = DeltaTable.forPath(spark, BRONZE_PATH).history(1).select("version").first()["version"]
            finalized_at = datetime.now(timezone.utc)
            DeltaTable.forPath(spark, MANIFEST_PATH).alias("t").update(
                condition=(
                    (F.col("t.dataset") == DATASET_NAME) &
                    (F.col("t.file_name") == file_name) &
                    (F.col("t.status") == "PROCESSING") &
                    (F.col("t.run_id") == run_id) &
                    (F.col("t.content_hash") == actual_hash)
                ),
                set={
                    "status": F.lit("SUCCESS"),
                    "batch_id": F.lit(batch_id),
                    "processed_at": F.lit(finalized_at).cast("timestamp"),
                    "last_success_at": F.lit(finalized_at).cast("timestamp"),
                    "status_updated_at": F.lit(finalized_at).cast("timestamp"),
                    "processed_content_hash": F.lit(actual_hash),
                    "source_record_count": F.lit(source_count).cast("long"),
                    "written_record_count": F.lit(written_count).cast("long"),
                    "rejected_record_count": F.lit(0).cast("long"),
                    "target_path": F.lit(BRONZE_PATH),
                    "target_delta_version": F.lit(version).cast("long"),
                    "error_message": F.lit(None).cast("string"),
                    "error_type": F.lit(None).cast("string"),
                    "last_error_at": F.lit(None).cast("timestamp"),
                    "lease_expires_at": F.lit(None).cast("timestamp"),
                }
            )
            final_row = (
                spark.read.format("delta").load(MANIFEST_PATH)
                .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
                .select("status", "run_id", "written_record_count", "target_delta_version").first()
            )
            assert final_row["status"] == "SUCCESS" and final_row["run_id"] == run_id, f"{file_name}: finalization check failed."
            results.append((file_name, "SUCCESS", source_count, written_count, version))
            print(f"SUCCESS {file_name}: source={source_count}, written={written_count}, target_before={target_before}, Delta version={version}")

        except Exception as exc:
            failed_at = datetime.now(timezone.utc)
            # Mark only the record still owned by this run as FAILED; preserve the error for investigation.
            DeltaTable.forPath(spark, MANIFEST_PATH).alias("t").update(
                condition=(
                    (F.col("t.dataset") == DATASET_NAME) &
                    (F.col("t.file_name") == file_name) &
                    (F.col("t.status") == "PROCESSING") &
                    (F.col("t.run_id") == run_id)
                ),
                set={
                    "status": F.lit("FAILED"),
                    "status_updated_at": F.lit(failed_at).cast("timestamp"),
                    "last_error_at": F.lit(failed_at).cast("timestamp"),
                    "error_message": F.lit(str(exc)[:4000]),
                    "error_type": F.lit(type(exc).__name__),
                    "lease_expires_at": F.lit(None).cast("timestamp"),
                }
            )
            print(f"FAILED {file_name}: {type(exc).__name__}: {exc}")
            results.append((file_name, "FAILED", None, None, None))

    result_schema = "file_name string, status string, source_count long, written_count long, delta_version long"
    results_df = spark.createDataFrame(results, result_schema)
    display(results_df)
    print("Run ID:", run_id)


DRY RUN complete. No writes performed.


In [0]:

# CELL 6 — FINAL INVENTORY AND MANIFEST CHECK (READ-ONLY)
bronze_final_df = spark.read.format("delta").load(BRONZE_PATH)
display(
    bronze_final_df.groupBy("_source_file").agg(
        F.count("*").alias("record_count"),
        F.countDistinct("product_id").alias("distinct_product_ids"),
        F.sum(F.when(F.col("product_id").isNull(), 1).otherwise(0)).alias("null_product_ids")
    ).orderBy("_source_file")
)

display(
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(F.col("dataset") == DATASET_NAME)
    .select("file_name", "status", "source_record_count", "written_record_count",
            "processed_content_hash", "target_delta_version", "lease_expires_at")
    .orderBy("file_name")
)
print("Total Bronze rows:", bronze_final_df.count())


_source_file,record_count,distinct_product_ids,null_product_ids
products_20261007.json,25000,25000,0
products_20261008.json,25000,25000,0


file_name,status,source_record_count,written_record_count,processed_content_hash,target_delta_version,lease_expires_at
products_20261007.json,SUCCESS,25000,25000,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,3,null
products_20261008.json,SUCCESS,25000,25000,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,0,2026-10-11T12:37:52.030Z


Total Bronze rows: 50000
